In [ ]:
import pandas as pd
import glob
import os
import requests
import rasterio
import numpy as np
import json

In [2]:


# 1. Căutăm fișierele (presupunem că sunt în folderul 'Dataseturi copaci')
cale_dataseturi = os.path.join('..', 'Dataseturi copaci')
fisiere_gasite = glob.glob(os.path.join(cale_dataseturi, "*.csv"))

mapare_coloane = {
    'Categorii de fructe': 'Specie',
    'Categorii de pomi fructiferi': 'Specie',
    'Macroregiuni  regiuni de dezvoltare si judete': 'Judet',
    'Ani': 'An'
}

lista_df = []

for fisier in fisiere_gasite:
    df = pd.read_csv(fisier)
    df.columns = df.columns.str.strip()
    df.rename(columns=mapare_coloane, inplace=True)
    
    # Standardizăm numele speciilor imediat după citire
    # Schimbăm "Meri" în "Mere" ca să se potrivească toate fișierele
    if 'Specie' in df.columns:
        df['Specie'] = df['Specie'].str.strip().replace('Meri', 'Mere')
    
    # Filtrăm DOAR pentru "Mere" chiar acum (economisim memorie)
    df = df[df['Specie'] == 'Mere']
    
    nume = os.path.basename(fisier).lower()
    if 'nr_pomi' in nume:
        df.rename(columns={'Valoare': 'Numar_Pomi'}, inplace=True)
    elif 'kg_pom' in nume:
        df.rename(columns={'Valoare': 'Kg_pe_Pom'}, inplace=True)
    elif 'tone_totale' in nume:
        df.rename(columns={'Valoare': 'Tone_Totale'}, inplace=True)

    # Convertim valorile la numere
    for col in ['Numar_Pomi', 'Kg_pe_Pom', 'Tone_Totale']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    cols_prezente = [c for c in ['Specie', 'Judet', 'An', 'Numar_Pomi', 'Kg_pe_Pom', 'Tone_Totale'] if c in df.columns]
    lista_df.append(df[cols_prezente])

if lista_df:
    # Unim datele filtrate pentru Meri
    df_merge = pd.concat(lista_df, ignore_index=True)
    
    # Consolidăm pe rânduri unice (Specie, Judet, An)
    # Folosim sum() pentru a colecta datele din fișiere diferite pe același rând
    df_final = df_merge.groupby(['Specie', 'Judet', 'An'], as_index=False).agg({
        'Numar_Pomi': lambda x: x.sum(min_count=1),
        'Kg_pe_Pom': lambda x: x.sum(min_count=1),
        'Tone_Totale': lambda x: x.sum(min_count=1)
    })


    print(f"Am extras datele pentru {len(df_final)} înregistrări cu Meri/Mere.")
    print(df_final.head())
    
    # Salvare rezultat
    df_final.to_csv('date_meri.csv', index=False)

Am extras datele pentru 1072 înregistrări cu Meri/Mere.
  Specie  Judet          An  Numar_Pomi  Kg_pe_Pom  Tone_Totale
0   Mere   Alba   Anul 2000    874496.0        8.0       6759.0
1   Mere   Alba   Anul 2001    828878.0       12.0       9728.0
2   Mere   Alba   Anul 2002    829665.0       11.0       8752.0
3   Mere   Alba   Anul 2003    828750.0       17.0      13811.0
4   Mere   Alba   Anul 2004    856253.0       36.0      30537.0


In [ ]:
df = pd.read_csv('date_meri.csv')


df['Max_Kg_Istoric'] = df.groupby('Judet')['Kg_pe_Pom'].transform('max')

def clasifica_rootstock(max_kg):
    if max_kg < 20:   return "Pitic"  # M9 — Pitic
    elif max_kg < 38: return "Semipitic"  # M26 — Semipitic
    elif max_kg < 55: return "Semiviguros"  # MM106 — Semiviguros
    else:             return "Viguros"  # Franc — Viguros

df['Rootstock_ID'] = df['Max_Kg_Istoric'].apply(clasifica_rootstock)

df = df.drop(columns=['Max_Kg_Istoric'])
df.to_csv('date_meri.csv', index=False)

print(df['Rootstock_ID'].value_counts().sort_index())

Rootstock_ID
1     47
2    625
3    325
4     75
Name: count, dtype: int64
